In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("cwd =", os.getcwd())


## Edit数据转化

In [ ]:
from logging import getLogger
from typing import Union
import torch
import os
from accelerate import Accelerator
from torch.utils.data import DataLoader

from genrec.dataset import AbstractDataset
from genrec.model import AbstractModel
from genrec.tokenizer import AbstractTokenizer
from genrec.utils import get_config, init_seed, init_logger, init_device, \
    get_dataset, get_tokenizer, get_model, get_trainer, log
import argparse
from genrec.utils import parse_command_line_args, get_pipeline

category = 'Video_Games'
if category == 'Video_Games':
		max_rows = 0.5
elif category == 'Office_Products':
		max_rows = 0.1
elif category == 'Cell_Phones_and_Accessories':
		max_rows = 0.1
elif category == 'Musical_Instruments': 
		max_rows = 0.5
elif category == 'Industrial_and_Scientific':
		max_rows = 0.5

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, default='TIGER', help='Model name')
    parser.add_argument('--dataset', type=str, default='AmazonReviews2023', help='Dataset name')
    parser.add_argument('--config_file', type=str, default="genrec/models/TIGER/config.yaml", help='Path to config file')
    return parser.parse_known_args()

args, unparsed_args = parse_args()
command_line_configs = {'pretrained_model_path': 'data/ckpt/TIGER_Sports_and_Outdoors/genrec_default-Jan-13-2026_00-19-06a35c.pth',
												'category': category,
												'max_rows': max_rows
												}
# import ipdb; ipdb.set_trace()

model_name = args.model
config = get_config(
		model_name=args.model,
		dataset_name=args.dataset,
		config_dict=command_line_configs,
		config_file = args.config_file
)

# Accelerator
project_dir = os.path.join(
		config['tensorboard_log_dir'],
		config["dataset"],
		config["model"]
)
accelerator = Accelerator(log_with='tensorboard', project_dir= project_dir)
config['accelerator'] =  accelerator

# Seed and Logger
init_seed(config['rand_seed'],  config['reproducibility'])

raw_dataset = get_dataset(args.dataset)(config)
split_datasets = raw_dataset.split()   # 这里把数据集划分好了 返回的是字典 key是'train','val','test' value是Dataset对象



In [ ]:
test_dataset = split_datasets["train"]
len(test_dataset)

In [ ]:
item2id, id2item = raw_dataset.id_mapping['item2id'], raw_dataset.id_mapping['id2item']

In [ ]:
import numpy as np
from datasets import Dataset

def build_cold_augmented(
    sent_embs: np.ndarray,           # (num_items_without_pad, 768) e.g. (18357, 768)
    split_datasets: dict,            # contains 'train', 'cold_test' etc.
    item2id: dict,                   # {'[PAD]':0, 'asin':1, ...}
    id2item: list,                   # ['[PAD]', 'asin1', ...]
    topk: int = 20,
    max_aug_per_target: int | None = None,   # 可选：限制每个target最多生成多少条增强样本，None=不限制
):
    train_ds = split_datasets["train"]
    cold_ds = split_datasets["cold_test"]

    # ----------------------------
    # 0) 训练集中出现过的 item 集合
    # ----------------------------
    train_items = set()
    for seq in train_ds["item_seq"]:
        train_items.update(seq)

    # 去掉 PAD（如果存在）
    train_items.discard("[PAD]")

    # 把训练 items 转成 sent_embs 的下标（id-1）
    train_emb_indices = []
    train_items_list = []
    for it in train_items:
        if it in item2id:
            iid = item2id[it]
            if iid > 0:
                emb_idx = iid - 1
                if 0 <= emb_idx < sent_embs.shape[0]:
                    train_emb_indices.append(emb_idx)
                    train_items_list.append(it)

    train_emb_indices = np.array(train_emb_indices, dtype=np.int64)
    # 对应的训练 embedding 矩阵 (N_train_items, dim)
    train_embs = sent_embs[train_emb_indices]

    # 归一化，便于余弦相似度 = dot
    train_norm = np.linalg.norm(train_embs, axis=1, keepdims=True)
    train_norm = np.maximum(train_norm, 1e-12)
    train_embs_norm = train_embs / train_norm

    # -----------------------------------------
    # 1) cold_test 里每条样本的 target item
    # -----------------------------------------
    cold_targets = []
    for seq in cold_ds["item_seq"]:
        if len(seq) > 0:
            cold_targets.append(seq[-1])

    # 去重（按出现顺序保留）
    seen = set()
    cold_targets_unique = []
    for t in cold_targets:
        if t not in seen:
            seen.add(t)
            cold_targets_unique.append(t)

    # -----------------------------------------
    # 2) 对每个 target，找训练集中 topk 相似 items
    #    输出：[{target:[sim_items...]}, ...]
    # -----------------------------------------
    target2sims = {}  # target -> list(sim_items)
    mapping_list = [] # [{target:[...]}...]

    # 为了速度：先建一个 item->emb_idx 的快速映射（target用）
    def item_to_embidx(it: str) -> int | None:
        iid = item2id.get(it, None)
        if iid is None or iid <= 0:
            return None
        emb_idx = iid - 1
        if 0 <= emb_idx < sent_embs.shape[0]:
            return emb_idx
        return None

    for target in cold_targets_unique:
        t_idx = item_to_embidx(target)
        if t_idx is None:
            # target 没 embedding / 不在映射中
            target2sims[target] = []
            mapping_list.append({target: []})
            continue

        t_vec = sent_embs[t_idx]
        t_norm = np.linalg.norm(t_vec)
        if t_norm < 1e-12:
            target2sims[target] = []
            mapping_list.append({target: []})
            continue

        t_vec_norm = t_vec / t_norm

        # cosine similarity with all train items
        sims = train_embs_norm @ t_vec_norm  # (N_train_items,)

        # 排除 target 本身（如果 target 也在训练 items 里）
        # train_items_list 与 sims 一一对应
        # 这里用 mask 更直观
        if target in train_items:
            print("有吗") # 没有
            mask = np.array([it != target for it in train_items_list], dtype=bool)
            sims_masked = sims[mask]
            items_masked = np.array(train_items_list, dtype=object)[mask]
        else:
            sims_masked = sims
            items_masked = np.array(train_items_list, dtype=object)

        k = min(topk, sims_masked.shape[0])
        if k <= 0:
            target2sims[target] = []
            mapping_list.append({target: []})
            continue

        # topk
        topk_idx = np.argpartition(-sims_masked, kth=k-1)[:k]
        # 排序（从大到小）
        topk_idx = topk_idx[np.argsort(-sims_masked[topk_idx])]

        sim_items = items_masked[topk_idx].tolist()
        target2sims[target] = sim_items
        mapping_list.append({target: sim_items})

    # -----------------------------------------
    # 3) 在训练集中找 sim_items 的出现位置，构造增强序列
    #    new_seq = prefix_before_sim_item + [target]
    # -----------------------------------------
    # 先建倒排：item -> [(user, seq, pos), ...]
    inverted = {}
    train_users = train_ds["user"]
    train_seqs = train_ds["item_seq"]

    for user, seq in zip(train_users, train_seqs):
        # seq 是 list
        for pos, it in enumerate(seq):
            inverted.setdefault(it, []).append((user, seq, pos))

    aug_users = []
    aug_item_seqs = []

    for target, sim_items in target2sims.items():
        produced = 0
        for sim_it in sim_items:
            occurrences = inverted.get(sim_it, [])
            for user, seq, pos in occurrences:
                prefix = seq[:pos]  # sim_it 之前的序列
                new_seq = prefix + [target]
                aug_users.append(user)
                aug_item_seqs.append(new_seq)

                produced += 1
                if max_aug_per_target is not None and produced >= max_aug_per_target:
                    break
            if max_aug_per_target is not None and produced >= max_aug_per_target:
                break

    cold_test_augmented = Dataset.from_dict({"user": aug_users, "item_seq": aug_item_seqs})

    # 把增强集塞回 split_datasets（不改原 dict 的话可先 copy）
    split_datasets["cold_test_augmented"] = cold_test_augmented

    return mapping_list, cold_test_augmented, split_datasets


## Tokenize

In [ ]:
tokenizer = get_tokenizer(model_name)(config, raw_dataset)
tokenized_datasets =  tokenizer.tokenize({"train":split_datasets["train"] })  ## 这里把数据集都tokenize了（变成了sids）
tokenized_datasets
# # Model
# with accelerator.main_process_first():
# 		model = get_model(model_name)(config,  raw_dataset,  tokenizer)


In [ ]:
tokenized_datasets['train']["input_ids"][0].shape

In [ ]:
# PROMPT

# 
# 已知 tokenized_datasets['cold_test_augmented'][0] 数据样例如下：{'input_ids': tensor([1025,   17,  378,  568,  770,    1,  472,  573,  770, 1026,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
#             0,    0,    0,    0,    0,    0,    0,    0,    0,    0]),
#  'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
#          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
#          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
#          0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
#  'labels': tensor([  68,  413,  598,  770, 1026])}
# 请构造成json文件格式如下：            
# edit_requests：[{
#                 "history": input_ids,#( list[int])
#                 "target_sids": labels,#( list[str])
#                 "case_id": str(len(edit_requests))
#             },...]

In [ ]:
import json
import torch

def build_edit_requests_json(tokenized_datasets, split="train"):
    ds = tokenized_datasets[split]
    edit_requests = []

    for i in range(len(ds)):
        ex = ds[i]

        input_ids = ex["input_ids"]
        attn_mask = ex.get("attention_mask", None)
        labels = ex["labels"]

        # 1) input_ids -> history (list[int])
        if isinstance(input_ids, torch.Tensor):
            input_ids = input_ids.tolist()

        # 可选：用 attention_mask 去掉 padding（推荐）
        # if attn_mask is not None:
        #     if isinstance(attn_mask, torch.Tensor):
        #         attn_mask = attn_mask.tolist()
        #     valid_len = int(sum(attn_mask))
        #     history = input_ids[:valid_len]
        # else:
        #     history = input_ids

        # 如果你想保留 padding（不截断），改成：
        history = input_ids

        # 2) labels -> target_sids (list[str])
        if isinstance(labels, torch.Tensor):
            labels = labels.tolist()
        target_sids = [str(x) for x in labels]

        edit_requests.append({
            "history": history,
            "target_sids": target_sids,
            "case_id": str(len(edit_requests))
        })

    # payload = {"edit_requests": edit_requests}

    return edit_requests
# # 用法：
# payload = build_edit_requests_json(tokenized_datasets, "train")
# print(payload[0])

# out_path="data/Edit/edit_COV.json"
# with open(out_path, "w", encoding="utf-8") as f:
# 		json.dump(payload, f, ensure_ascii=False, indent=2)



In [ ]:
# 用法：
payload = build_edit_requests_json(tokenized_datasets, "train")
print(payload[0])

out_path="data/Edit/edit_COV.json"
with open(out_path, "w", encoding="utf-8") as f:
		json.dump(payload, f, ensure_ascii=False, indent=2)


In [ ]:

print([target["history"] for target in payload[0:2]])

In [ ]:

print([target["history"] for target in payload[0:2]])